# 00b — Statistiques descriptives par segment

On reprend la base obtenue à la fin du notebook 00 (périmètre final, dossiers joints unifiés, segment SA / SB) et on décrit les variables séparément dans chaque segment, puisque chaque segment aura son propre modèle.

Pour chaque segment :
1. valeurs manquantes ;
2. variables constantes ou presque ;
3. distributions et valeurs extrêmes ;
4. variables qualitatives ;
5. redondances entre variables ;
6. pouvoir discriminant (IV) ;
7. synthèse : statut provisoire de chaque variable avant la discrétisation, et export Excel.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import polars as pl
from utils import binning, exploration

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(80)

ID, TARGET, DATE, SEGMENT = "gdt", "ddefaut_ndb", "datdelhis", "segment"
BASE = Path("data/processed/base_segmentee.parquet")

# Chargement optimisé depuis le fichier Parquet
base = pl.read_parquet(BASE)

# Répartition globale des segments
base.group_by(SEGMENT).agg(
    n=pl.len(), n_defaut=pl.col(TARGET).sum(),
    taux_defaut=pl.col(TARGET).mean()
).sort(SEGMENT)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


segment,n,n_defaut,taux_defaut
str,u32,f64,f64
"""SA_sans_engagement""",94201,515.0,0.005467
"""SB_avec_engagement""",169489,1779.0,0.010496


In [4]:
valeurs_segments = sorted(base[SEGMENT].unique().to_list())
print("Segments disponibles :", valeurs_segments)

# Sélection du premier segment (Segment 1)
nom_seg1 = valeurs_segments[0]
df_seg1 = base.filter(pl.col(SEGMENT) == nom_seg1)

print(f"Segment analysé : {nom_seg1}")
print(f"Dimensions : {df_seg1.shape[0]} lignes, {df_seg1.shape[1]} colonnes")
print(f"Taux de défaut observé : {df_seg1[TARGET].mean():.2%}")

Segments disponibles : ['SA_sans_engagement', 'SB_avec_engagement']
Segment analysé : SA_sans_engagement
Dimensions : 94201 lignes, 173 colonnes
Taux de défaut observé : 0.55%


In [5]:
EXCLURE = {ID, TARGET, DATE, SEGMENT, "has_engagement", "SelectionProb", "SamplingWeight", "type_PM_PP", "code_marche_v2"}

VARS = [c for c, t in df_seg1.schema.items() if c not in EXCLURE and not t.is_temporal()]
VARS_NUM = [c for c in VARS if df_seg1.schema[c].is_numeric()]
VARS_QUALI = [c for c in VARS if c not in VARS_NUM]

print(f"{len(VARS)} variables analysées : {len(VARS_NUM)} numériques, {len(VARS_QUALI)} qualitatives")

161 variables analysées : 152 numériques, 9 qualitatives


# Analyse des valeurs manquantes

In [7]:
# 1. Taux de valeurs manquantes par variable
stats_manquants = df_seg1.select(
    [(pl.col(c).null_count() / pl.len()).alias(c) for c in VARS]
).transpose(include_header=True, header_name="variable", column_names=["pct_manquant"])

print("Top 10 variables avec le plus de manquants :")
print(stats_manquants.sort("pct_manquant", descending=True).head(10))

# 2. Statistiques des variables numériques (moyenne, écart-type, quartiles...)
print("\nStatistiques des variables numériques :")
df_seg1.select(VARS_NUM).describe()

Top 10 variables avec le plus de manquants :
shape: (10, 2)
┌─────────────────────────────────┬──────────────┐
│ variable                        ┆ pct_manquant │
│ ---                             ┆ ---          │
│ str                             ┆ f64          │
╞═════════════════════════════════╪══════════════╡
│ AGE_PM_EN_MOIS                  ┆ 1.0          │
│ RATIO_LITIGES_vs_CA             ┆ 1.0          │
│ LISTE_CODNAF2                   ┆ 0.994788     │
│ CRTOC_TOK_0010                  ┆ 0.990127     │
│ CODNAF2                         ┆ 0.989745     │
│ LISTE_CODETAJUR                 ┆ 0.986412     │
│ MVTAFF_STABILITE_2T_vs_12M      ┆ 0.985881     │
│ CODITDBDF_PRO                   ┆ 0.985106     │
│ CODETAJUR                       ┆ 0.972824     │
│ RATIO_NBJDEB_NONAUTO_SUR_NBJDEB ┆ 0.885277     │
└─────────────────────────────────┴──────────────┘

Statistiques des variables numériques :


statistic,nb_tiers,nb_crt_distincts,engagement,bilan,horsbilan,nb_AC_distincts,nb_agence_gestion_distincts,nb_produits_distincts,nb_credit_immo,nb_credit_immo__sans_garantie,nb_credit_immo__crelog,nb_credit_immo__hypotheque,nb_credit_immo__autre_OCM,nb_credit_immo__contrat_vie,nb_credit_immo__autre_garantie,engagement_immo,engagement_immo__sans_garantie,engagement_immo__crelog,engagement_immo__hypotheque,engagement_immo__autre_OCM,engagement_immo__contrat_vie,engagement_immo__autre_garantie,nb_credit_MLT,nb_credit_MLT__sans_garantie,nb_credit_MLT__interfimo,nb_credit_MLT__autre_OCM,nb_credit_MLT__PGE,nb_credit_MLT__autre_garantie,engagement_MLT,engagement_MLT__sans_garantie,engagement_MLT__interfimo,engagement_MLT__autre_OCM,engagement_MLT__PGE,engagement_MLT__autre_garantie,nb_DAV,nb_CB_diff,nb_credit_perso,nb_credit_flex,nb_credit_LCP,nb_credit_revolving,nb_credit_bail,nb_credit_tresorerie,nb_credit_garantie_emise,nb_credit_autre,nb_Neiertz,nb_PGE,engagement_DAV,engagement_CB_diff,engagement_credit_perso,engagement_credit_flex,engagement_credit_LCP,engagement_credit_revolving,engagement_credit_bail,engagement_credit_tresorerie,engagement_credit_garantie_emise,engagement_credit_autre,engagement_credit_neiertz,engagement_credit_PGE,solde_DAV_moyen,solde_debiteur,solde_crediteur,anciennete_jour_sain_M0,NBJ_ARR_SS,ARR_GDT,Dfo_max_Mm1_Mm3,Dfo_max_Mm4_Mm6,Dfo_max_Mm7_Mm9,Dfo_max_Mm10_Mm12,Dfo_max_Mm13_Mm18,Dfo_max_Mm19_Mm24,Dfo_max_Mm25_Mm36,Dfo_max_Mm37_Mm48,NBJ_ARR_SS_max_Mm1_Mm3,NBJ_ARR_SS_max_Mm4_Mm6,NBJ_ARR_SS_max_Mm7_Mm9,NBJ_ARR_SS_max_Mm10_Mm12,NBJ_ARR_SS_max_Mm13_Mm18,NBJ_ARR_SS_max_Mm19_Mm24,NBJ_ARR_SS_max_Mm25_Mm36,NBJ_ARR_SS_max_Mm37_Mm48,ANCIENNETE_modif,AGE_PP_EN_MOIS,AGE_PM_EN_MOIS,CODACVPRO_modif,MNT_HA102,CRTAD_IND_0038,CRTOC_IND_0164,CRTOC_SLD_MOY,CRTOP_AG_IND_0038,CRTAD_AG_SOLDE_T,CRTAD_AG_SOLDE_T1,CRTAD_AG_SOLDE_T2,CRTAD_AG_SOLDE_T3,CRTAD_AG_SOLDE_T4,MINSOLDE_PAR,CRTAD_AG_SOLDE_IND_0009,CRTAD_IND_0015,CRTAD_AG_NBJDE_BA,CRTAD_AG_NBJDE_BB,CRTAD_AG_NBJDE_BC,CRTOC_IND_0301,CRTOC_AG_NBJDEB_IND_0015,CRTOC_AG_LIGDEB_IND_0015,CRTOC_AG_LIGCRE_IND_0009,CRTOC_AG_NBJDEB_2T_IND_0015,CRTOC_AG_LIGDEB_2T_IND_0015,CRTOC_AG_NBJCRE_2T_IND_0009,CRTOC_AG_LIGCRE_2T_IND_0009,TOTAL_LITIGES_PRO_IND_0209,CRTAD_IND_0062,CRTAD_IND_0042,CRTAD_AG_NBECR_A,CRTAD_AG_NBECR_B,NBECR_PAR,CRTAD_AG_MVTAFF_IND_0010,CRTOC_IND_0010,CRTOC_MOY_MVTAFF_IND_0010,CRTOC_AG_MVTAFF_IND_0010,CRTOC_AG_MVTAFF_2T_IND_0010,RATIO1_PRO,RATIO3_PRO,ENCOURS_PAR,CRTAE_AG_SLDEPA,MNTAUTCRT_AG_EPTE_PAR,CRTAD_IND_0205,CRTAV_IND_0165,CRTAT_IND_0205,CRTAR_IND_0036,CRTPI_IND_0036,TOTAL_TITRES_PRO_IND_0205,TOTAL_EPS_PRO_IND_0208,NB_CCT_PRO,TX_UTIL_CCT_PRO,SOLDE_EVOLUTION_T_vs_T1,SOLDE_EVOLUTION_T_vs_T4,SOLDE_RATIO_T_vs_HIST,SOLDE_PENTE_5T,SOLDE_PENTE_4T,SOLDE_PENTE_3T,NBJDE_RATIO_A_vs_A1,RATIO_SOLDE_MOYEN_SUR_EPARGNE,RATIO_NBJDEB_NONAUTO_SUR_NBJDEB,RATIO_MOUV_AFF_SUR_EPARGNE,SOLDE_VOLATILITE_5T,SOLDE_VOLATILITE_3T,NBECR_EVOLUTION_A_vs_A1,MVTAFF_STABILITE_2T_vs_12M,PART_EPARGNE_DANS_AVOIRS,RATIO_LITIGES_vs_CA,RATIO_LITIGES_vs_AVOIRS,RATIO_ECR,cod_axe_unite
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0,94201.0